# Baseline experiments
**Literature-Grounded Research Idea Generation Using Specialized Multi-Agent Reviewers**

FAST-NUCES Islamabad. We test whether structured review changes proposal-level judged quality.
ResearchAgent inspires retrieval, critique and refinement; this generic baseline is an approximation,
not a reproduction. ResearchAgent already uses criterion-specific reviewers.

**Execution status:** The notebook reads real paper records and available results. Live experiments
require a locally configured Gemini key and network access. Missing findings remain explicitly absent.
Unit-test fixtures never appear as generated examples.


**Corrected live pilot (`smoke_v2`):** All twelve final-condition rows have accepted independent ratings, and all sixteen distinct judged proposals passed validation. One generic-review run stopped early on a wrong-role response. The full panel has higher mean feasibility, clarity and grounding but lower mean novelty than one-shot in this two-seed sample. These are preliminary proposal-level ratings; no statistical superiority or human validation is claimed. The earlier quotation-failure pilot remains preserved in `results/smoke`.


## Dataset and LLM configuration
Two fixed real papers are used for smoke testing; the separate 20-paper manifest is a purposive
AI/NLP ideation/review sample. It is not a random NLP benchmark. Every condition shares its initial
proposal, literature, constraints and model settings. See `data/seed_provenance.json`.


In [1]:
from pathlib import Path
import sys, os, json
ROOT = Path.cwd() if (Path.cwd() / 'src' / 'research_review').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from research_review.config import Settings, budget
from research_review.schemas import Paper, Run
from research_review.storage import load_json
from research_review.literature_retrieval import create_retriever
from research_review.llm import GeminiClient
from research_review.experiments import run_study
settings = Settings.from_env()
study = ROOT / 'results' / os.getenv('RESEARCH_REVIEW_STUDY', 'smoke_v2')
seeds = [Paper.model_validate(p) for p in load_json(ROOT / 'data' / 'smoke_seeds.json')]
retriever = create_retriever(settings)
RUN_LIVE = os.getenv('RUN_LIVE_EXPERIMENTS') == '1'
print('Model:', settings.model, '| rounds:', settings.rounds)
print('Credentials configured:', bool(os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')))
print('Live execution opt-in:', RUN_LIVE)
print('Fixed smoke seeds:', [(p.paper_id, p.year, p.title) for p in seeds])


Model: gemini-3.5-flash-lite | rounds: 3
Credentials configured: True
Live execution opt-in: False
Fixed smoke seeds: [('2025.naacl-long.342', 2025, 'ResearchAgent: Iterative Research Idea Generation over Scientific Literature with Large Language Models'), ('2024.acl-long.18', 2024, 'SciMON: Scientific Inspiration Machines Optimized for Novelty')]


In [2]:
print(json.dumps(budget(len(seeds), settings.rounds), indent=2))
print('Free-tier estimates assume the account is on the free tier; quotas are account-specific.')

{
  "seeds": 2,
  "rounds": 3,
  "generation_review_calls": 122,
  "judge_calls_upper_bound": 16,
  "total_calls_upper_bound": 138,
  "with_retries_upper_bound": 414,
  "planning_input_tokens": 897000,
  "planning_output_tokens": 193200,
  "note": "Planning token assumptions, not measured usage. Six natural-budget configurations; no equal-cost causal claim."
}
Free-tier estimates assume the account is on the free tier; quotas are account-specific.


## One-shot and generic iterative review
One-shot returns the shared initial proposal. Generic review uses one general critique and one revision
per round, up to three rounds. Specialist conditions consume more calls; the primary comparison uses
natural budgets and cannot isolate role specialization from extra inference. Repeated independent
replicates and budget-matched controls would strengthen the study.

To execute live cells, set `RUN_LIVE_EXPERIMENTS=1` before launching Jupyter. Default cells only inspect
saved artifacts. The CLI runs all six conditions in randomized order and is preferred for the main study.


In [3]:
if RUN_LIVE:
    client = GeminiClient(settings, study / 'api_audit')
    run_study(seeds, client, retriever, settings, study, configurations=['one_shot', 'generic'])
else:
    print('Live baseline execution disabled; inspecting saved results.')


Live baseline execution disabled; inspecting saved results.


In [4]:
runs = [Run.model_validate(load_json(p)) for p in (study / 'runs').glob('*.json') if not p.name.endswith('.partial.json')]
print('Saved live runs:', sum(r.provenance == 'live_gemini' for r in runs))
if not runs:
    print('NOT RUN: No live generated proposals are available. Dataset and software checks are not experimental findings.')


Saved live runs: 12


In [5]:
for r in runs:
    if r.configuration in ('one_shot', 'generic'):
        print('Condition:', r.configuration, '| seed:', r.seed_id, '| stop:', r.stop_reason)
        print(json.dumps(r.final_idea.model_dump(), indent=2))
        print('Review/revision API attempts excluding shared initial:', len(r.api_events))


Condition: generic | seed: 2024.acl-long.18 | stop: all_reviewers_failed
{
  "title": "Latent-Probed Multi-Agent Iterative Refinement for Novel Scientific Idea Generation",
  "research_problem": "How can multi-agent language model dialogues be optimized to generate scientific research ideas with high novelty while mitigating the systematic bias of language models toward generating ideas of mediocre novelty?",
  "research_gap": "Prior work explores multi-agent LLM dialogues for scientific ideation and latent novelty probing independently, but lacks an integrated framework where hidden-state novelty probing is used inside an ideation-critique-revision loop to dynamically steer agents away from medium-novelty biases during generation without requiring model weight training.",
  "methodology": "We propose a unified framework combining multi-agent dialogue loops with latent-space novelty probing. Specifically, we instantiate a generator agent and a critic agent. During the critique and revi

## Interpretation
Use independently judged scores from Notebook 3. A polished revision or a high reviewer score does
not establish improvement. If no live artifacts were printed above, baseline effectiveness has not
been measured. Initial proposals, raw API records, and revisions are saved by the reusable runner.
